# Problem 15: Rectified Flow on 2D Toy Data (30 pts)

**Part 1 (10 pts):** Run the provided 2D toy example end-to-end, visualize learned trajectories.

**Part 2 (10 pts):** Apply RF to new 2D datasets (from HW4: `2spirals`, `8gaussians`, `rings`), experiment with hyperparameters.

**Part 3 (10 pts):** Implement the RF loss manually (without `get_loss`), show consistency with the built-in version.

In [ ]:
!git clone https://github.com/aldookware/rectified-flow.git
%cd rectified-flow/

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from rectified_flow.utils import set_seed
from rectified_flow.utils import visualize_2d_trajectories_plotly
from rectified_flow.rectified_flow import RectifiedFlow

set_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
## Part 1 (10 pts): Run the Provided 2D Toy Example

We train a 1-Rectified Flow on the default two-Gaussian-mixture (GMM) dataset, visualize
trajectories, and then apply reflow to obtain 2-Rectified Flow for one-step generation.

### Generate $\pi_0$ and $\pi_1$

In [ ]:
from rectified_flow.datasets.toy_gmm import TwoPointGMM

n_samples = 50000
pi_0 = TwoPointGMM(x=0.0, y=7.5, std=0.5, device=device)
pi_1 = TwoPointGMM(x=15.0, y=7.5, std=0.5, device=device)
D0 = pi_0.sample([n_samples])
D1, labels = pi_1.sample_with_labels([n_samples])

plt.figure(figsize=(4, 4))
plt.title(r'Samples from $\pi_0$ and $\pi_1$')
plt.scatter(D0[:, 0].cpu(), D0[:, 1].cpu(), alpha=0.3, s=2, label=r'$\pi_0$')
plt.scatter(D1[:, 0].cpu(), D1[:, 1].cpu(), alpha=0.3, s=2, label=r'$\pi_1$')
plt.legend()
plt.show()

### Visualize Straight Interpolation

In [ ]:
x_0 = pi_0.sample([500])
x_0_upper = x_0.clone()
x_0_upper[:, 1] = torch.abs(x_0_upper[:, 1])
x_0_lower = x_0.clone()
x_0_lower[:, 1] = -torch.abs(x_0_lower[:, 1])

x_1_upper = pi_1.sample([500])
x_1_lower = pi_1.sample([500])

interp_upper, interp_lower = [], []
for t in np.linspace(0, 1, 100):
    interp_upper.append((1 - t) * x_0_upper + t * x_1_upper)
    interp_lower.append((1 - t) * x_0_lower + t * x_1_lower)

visualize_2d_trajectories_plotly(
    {"upper": interp_upper, "lower": interp_lower},
    D1_gt_samples=torch.cat([x_1_upper, x_1_lower], dim=0),
    num_trajectories=100,
    title="Straight Interpolation",
)

### Train 1-Rectified Flow (Unconditional)

In [ ]:
from rectified_flow.models.toy_mlp import MLPVelocity

model = MLPVelocity(2, hidden_sizes=[128, 128, 128]).to(device)

rectified_flow = RectifiedFlow(
    data_shape=(2,),
    velocity_field=model,
    interp="straight",
    source_distribution=pi_0,
    device=device,
)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
batch_size = 1024
losses_1rf = []

for step in range(5000):
    optimizer.zero_grad()
    idx = torch.randperm(n_samples)[:batch_size]
    x_0 = D0[idx].to(device)
    x_1 = D1[idx].to(device)

    loss = rectified_flow.get_loss(x_0, x_1)
    loss.backward()
    optimizer.step()
    losses_1rf.append(loss.item())

    if step % 1000 == 0:
        print(f"Step {step}, Loss: {loss.item():.4f}")

plt.figure(figsize=(8, 3))
plt.plot(losses_1rf)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("1-RF Training Loss (GMM)")
plt.grid(True, alpha=0.3)
plt.show()

### Visualize 1-RF Trajectories (100 steps)

In [ ]:
from rectified_flow.samplers import EulerSampler

euler_sampler_1rf = EulerSampler(rectified_flow=rectified_flow, num_steps=100)

traj_upper = euler_sampler_1rf.sample_loop(x_0=x_0_upper).trajectories
traj_lower = euler_sampler_1rf.sample_loop(x_0=x_0_lower).trajectories

visualize_2d_trajectories_plotly(
    {"upper": traj_upper, "lower": traj_lower},
    D1_gt_samples=D1[:1000],
    num_trajectories=200,
    title="1-Rectified Flow Trajectories (100 steps)",
)

### 1-RF One-Step Generation

Since 1-RF trajectories are curved, one-step generation $\hat{X}_1 = X_0 + v(X_0, 0)$ is inaccurate.

In [ ]:
euler_sampler_1rf.sample_loop(num_steps=1, seed=0)

visualize_2d_trajectories_plotly(
    {"1rf one-step": euler_sampler_1rf.trajectories},
    D1[:1000],
    num_trajectories=200,
    title="1-RF One-Step Generation",
)

### Reflow: 2-Rectified Flow

Reflow straightens the trajectories by training on the coupling $(Z_0, Z_1)$ simulated from 1-RF.
After reflow, one-step generation becomes much more accurate.

In [ ]:
# Generate (Z_0, Z_1) coupling from 1-RF
Z_0 = rectified_flow.sample_source_distribution(batch_size=50000)
Z_1 = euler_sampler_1rf.sample_loop(x_0=Z_0, num_steps=1000).trajectories[-1]

In [ ]:
# Re-train on the reflowed coupling
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
losses_2rf = []

for step in range(5000):
    optimizer.zero_grad()
    idx = torch.randperm(n_samples)[:batch_size]
    x_0 = Z_0[idx].to(device)
    x_1 = Z_1[idx].to(device)

    loss = rectified_flow.get_loss(x_0, x_1)
    loss.backward()
    optimizer.step()
    losses_2rf.append(loss.item())

    if step % 1000 == 0:
        print(f"Step {step}, Loss: {loss.item():.4f}")

plt.figure(figsize=(8, 3))
plt.plot(losses_2rf)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("2-RF Training Loss (Reflow)")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
euler_sampler_2rf = EulerSampler(rectified_flow=rectified_flow, num_samples=1000)

# 100-step
euler_sampler_2rf.sample_loop(num_steps=100, seed=0)
visualize_2d_trajectories_plotly(
    {"2rf": euler_sampler_2rf.trajectories},
    D1[:1000], num_trajectories=200,
    title="2-RF Trajectories (100 steps)",
)

In [ ]:
# One-step — much better after reflow!
euler_sampler_2rf.sample_loop(num_steps=1, seed=0)
visualize_2d_trajectories_plotly(
    {"2rf one-step": euler_sampler_2rf.trajectories},
    D1[:1000], num_trajectories=200,
    title="2-RF One-Step Generation",
)

### Part 1 Observations

**1-RF trajectories (100 steps):** The learned flow successfully transports $\pi_0$ to $\pi_1$. Trajectories are curved due to the averaging $v(z,t) = \mathbb{E}[\dot{X}_t | X_t = z]$ at intersection points. The upper/lower subsets evolve separately — the flow "causalizes" the interpolation.

**1-RF one-step:** Poor quality — trajectories are too curved for a single Euler step to be accurate.

**2-RF (reflow):** After reflow, trajectories become nearly straight. One-step generation is now accurate because the coupling $(Z_0, Z_1)$ is nearly deterministic, eliminating trajectory crossings.

---
## Part 2 (10 pts): Apply RF to HW4 2D Toy Datasets

We replace the GMM data with datasets from HW4: `2spirals`, `8gaussians`, and `rings`.
We experiment with hyperparameters (learning rate, training steps, batch size) and compare results.

In [ ]:
# --- HW4 dataset generator (from GAN_2d_toy.ipynb) ---
def generate_data(data: str, batch_size: int = 200, device: str = "cpu") -> torch.Tensor:
    if data == "rings":
        n4 = n3 = n2 = batch_size // 4
        n1 = batch_size - n4 - n3 - n2
        angles = [torch.linspace(0, 2*np.pi, n+1, device=device)[:-1] for n in [n1, n2, n3, n4]]
        radii = [0.25, 0.50, 0.75, 1.00]
        circs = [torch.stack([torch.cos(a), torch.sin(a)], dim=1) * r for a, r in zip(angles, radii)]
        X = torch.cat(circs, dim=0) * 3.0 + torch.randn(batch_size, 2, device=device) * 0.08
        return X[torch.randperm(X.size(0), device=device)].float()

    elif data == "8gaussians":
        scale = 4.0
        centers = torch.tensor([
            [0,1],[-1/np.sqrt(2),1/np.sqrt(2)],[-1,0],[-1/np.sqrt(2),-1/np.sqrt(2)],
            [0,-1],[1/np.sqrt(2),-1/np.sqrt(2)],[1,0],[1/np.sqrt(2),1/np.sqrt(2)]
        ], dtype=torch.float32, device=device) * scale
        idx = torch.randint(0, 8, (batch_size,), device=device)
        X = (torch.randn(batch_size, 2, device=device) * 0.5 + centers[idx]) / 1.414
        return X[torch.randperm(X.size(0), device=device)].float()

    elif data == "2spirals":
        half = batch_size // 2
        n = torch.sqrt(torch.rand(half, 1, device=device)) * (3 * np.pi)
        d1x = -torch.cos(n) * n + torch.rand(half, 1, device=device) * 0.5
        d1y =  torch.sin(n) * n + torch.rand(half, 1, device=device) * 0.5
        spiral1 = torch.cat([d1x, d1y], dim=1)
        X = torch.cat([spiral1, -spiral1], dim=0) / 3.0 + torch.randn(batch_size, 2, device=device) * 0.1
        return X[torch.randperm(X.size(0), device=device)][:batch_size].float()

    else:
        raise ValueError(f"Unknown dataset: {data}")

# Visualize all three datasets
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, name in zip(axes, ["2spirals", "8gaussians", "rings"]):
    data = generate_data(name, batch_size=2000, device="cpu")
    ax.scatter(data[:, 0], data[:, 1], s=2, alpha=0.5)
    ax.set_title(name)
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
def train_rf_on_dataset(dataset_name, n_samples=50000, lr=1e-3, batch_size=1024,
                        n_steps=5000, hidden_sizes=[128, 128, 128]):
    """Train 1-RF on a given 2D dataset and return model, losses, and generated samples."""
    # Generate target data
    D1_hw4 = generate_data(dataset_name, batch_size=n_samples, device=device)

    # Source: standard Gaussian
    D0_hw4 = torch.randn(n_samples, 2, device=device)

    model_hw4 = MLPVelocity(2, hidden_sizes=hidden_sizes).to(device)

    # Use standard Gaussian as source distribution
    source_dist = torch.distributions.MultivariateNormal(
        torch.zeros(2, device=device), torch.eye(2, device=device)
    )

    rf_hw4 = RectifiedFlow(
        data_shape=(2,),
        velocity_field=model_hw4,
        interp="straight",
        source_distribution=source_dist,
        device=device,
    )

    optimizer = torch.optim.Adam(model_hw4.parameters(), lr=lr)
    losses = []

    for step in range(n_steps):
        optimizer.zero_grad()
        idx = torch.randperm(n_samples, device=device)[:batch_size]
        x_0 = D0_hw4[idx]
        x_1 = D1_hw4[idx]

        loss = rf_hw4.get_loss(x_0, x_1)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    # Generate samples
    sampler = EulerSampler(rectified_flow=rf_hw4, num_steps=100, num_samples=2000)
    sampler.sample_loop(seed=42)
    generated = sampler.trajectories[-1].cpu()

    return losses, generated, D1_hw4.cpu()

### Train RF on each HW4 dataset (default hyperparameters)

In [ ]:
datasets = ["2spirals", "8gaussians", "rings"]
results = {}

for name in datasets:
    print(f"\n=== Training on {name} ===")
    losses, generated, real = train_rf_on_dataset(name)
    results[name] = {"losses": losses, "generated": generated, "real": real}
    print(f"Final loss: {losses[-1]:.4f}")

# Plot loss curves side by side
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, datasets):
    ax.plot(results[name]["losses"])
    ax.set_title(f"{name} — Training Loss")
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Plot real vs generated
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for j, name in enumerate(datasets):
    real = results[name]["real"]
    gen = results[name]["generated"]
    axes[0, j].scatter(real[:2000, 0], real[:2000, 1], s=2, alpha=0.5)
    axes[0, j].set_title(f"{name} — Real")
    axes[0, j].set_aspect('equal')
    axes[1, j].scatter(gen[:, 0], gen[:, 1], s=2, alpha=0.5, color='tab:orange')
    axes[1, j].set_title(f"{name} — RF Generated")
    axes[1, j].set_aspect('equal')
plt.tight_layout()
plt.show()

### Hyperparameter Sweep

We test different learning rates, training steps, and batch sizes on `2spirals`.

In [ ]:
hp_configs = [
    {"lr": 1e-4, "n_steps": 5000, "batch_size": 1024, "label": "lr=1e-4"},
    {"lr": 1e-3, "n_steps": 5000, "batch_size": 1024, "label": "lr=1e-3 (default)"},
    {"lr": 5e-3, "n_steps": 5000, "batch_size": 1024, "label": "lr=5e-3"},
    {"lr": 1e-3, "n_steps": 10000, "batch_size": 1024, "label": "10k steps"},
    {"lr": 1e-3, "n_steps": 5000, "batch_size": 256, "label": "bs=256"},
    {"lr": 1e-3, "n_steps": 5000, "batch_size": 4096, "label": "bs=4096"},
]

hp_results = {}
for cfg in hp_configs:
    label = cfg["label"]
    print(f"Training: {label}")
    losses, generated, real = train_rf_on_dataset(
        "2spirals", lr=cfg["lr"], n_steps=cfg["n_steps"], batch_size=cfg["batch_size"]
    )
    hp_results[label] = {"losses": losses, "generated": generated, "real": real}

# Loss curves
plt.figure(figsize=(10, 5))
for label, res in hp_results.items():
    plt.plot(res["losses"], label=label, alpha=0.8)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Hyperparameter Sweep on 2spirals")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Generated samples comparison
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for idx, (label, res) in enumerate(hp_results.items()):
    r, c = idx // 3, idx % 3
    gen = res["generated"]
    axes[r, c].scatter(gen[:, 0], gen[:, 1], s=2, alpha=0.5)
    axes[r, c].set_title(label)
    axes[r, c].set_aspect('equal')
plt.suptitle("2spirals — Generated Samples by Hyperparameter Config", fontsize=14)
plt.tight_layout()
plt.show()

### Part 2 Observations

**Dataset comparison:** Rectified Flow captures all three datasets well. `8gaussians` is the easiest (isolated modes), `rings` requires learning concentric structure, and `2spirals` is the most challenging due to the interleaved spiral geometry.

**Learning rate:** `lr=1e-3` works well as a default. `lr=1e-4` converges too slowly in 5k steps. `lr=5e-3` may overshoot initially but can converge faster.

**Training steps:** 10k steps provides noticeably better results on `2spirals` — the fine spiral structure benefits from longer training.

**Batch size:** Larger batches (4096) give smoother loss curves and slightly better results due to lower gradient variance. Smaller batches (256) are noisier but still converge.

---
## Part 3 (10 pts): Manual RF Loss Implementation

We implement the rectified flow loss from scratch without using `get_loss`:

$$\mathcal{L}(v_\theta) = \mathbb{E}_{t \sim U[0,1],\, (x_0, x_1) \sim (\pi_0, \pi_1)} \left[\|(x_1 - x_0) - v_\theta(x_t, t)\|^2\right]$$

where $x_t = t \cdot x_1 + (1 - t) \cdot x_0$ (straight interpolation).

We train two models side by side — one with `get_loss`, one with our manual loss — and compare.

In [ ]:
set_seed(42)

# --- Model A: using get_loss (built-in) ---
model_A = MLPVelocity(2, hidden_sizes=[128, 128, 128]).to(device)
rf_A = RectifiedFlow(
    data_shape=(2,), velocity_field=model_A, interp="straight",
    source_distribution=pi_0, device=device,
)
opt_A = torch.optim.Adam(model_A.parameters(), lr=1e-3)

# --- Model B: manual loss ---
model_B = MLPVelocity(2, hidden_sizes=[128, 128, 128]).to(device)
# Copy initial weights so both start identically
model_B.load_state_dict(model_A.state_dict())
opt_B = torch.optim.Adam(model_B.parameters(), lr=1e-3)

In [ ]:
losses_A, losses_B = [], []

for step in range(5000):
    # Same batch for both
    idx = torch.randperm(n_samples)[:batch_size]
    x_0_batch = D0[idx].to(device)
    x_1_batch = D1[idx].to(device)
    t = torch.rand(batch_size, device=device)  # t ~ U[0,1]

    # --- Model A: built-in get_loss ---
    opt_A.zero_grad()
    loss_A = rf_A.get_loss(x_0_batch, x_1_batch, t=t)
    loss_A.backward()
    opt_A.step()
    losses_A.append(loss_A.item())

    # --- Model B: manual loss ---
    opt_B.zero_grad()
    # Straight interpolation: x_t = t * x_1 + (1 - t) * x_0
    t_expand = t.unsqueeze(1)  # (B, 1)
    x_t = t_expand * x_1_batch + (1 - t_expand) * x_0_batch
    # Target velocity: dot_x_t = x_1 - x_0
    dot_x_t = x_1_batch - x_0_batch
    # Model prediction
    v_pred = model_B(x_t, t)
    # MSE loss
    loss_B = ((v_pred - dot_x_t) ** 2).mean()
    loss_B.backward()
    opt_B.step()
    losses_B.append(loss_B.item())

    if step % 1000 == 0:
        print(f"Step {step}: get_loss={loss_A.item():.4f}  manual={loss_B.item():.4f}")

In [ ]:
# Compare loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(losses_A, label='get_loss (built-in)', alpha=0.7)
axes[0].plot(losses_B, label='Manual loss', alpha=0.7)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Smoothed version
window = 100
smooth_A = np.convolve(losses_A, np.ones(window)/window, mode='valid')
smooth_B = np.convolve(losses_B, np.ones(window)/window, mode='valid')
axes[1].plot(smooth_A, label='get_loss (smoothed)', alpha=0.8)
axes[1].plot(smooth_B, label='Manual (smoothed)', alpha=0.8)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Loss')
axes[1].set_title('Smoothed Loss Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compare generated samples
rf_B = RectifiedFlow(
    data_shape=(2,), velocity_field=model_B, interp="straight",
    source_distribution=pi_0, device=device,
)

sampler_A = EulerSampler(rectified_flow=rf_A, num_steps=100, num_samples=2000)
sampler_B = EulerSampler(rectified_flow=rf_B, num_steps=100, num_samples=2000)

sampler_A.sample_loop(seed=0)
sampler_B.sample_loop(seed=0)

gen_A = sampler_A.trajectories[-1].cpu()
gen_B = sampler_B.trajectories[-1].cpu()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].scatter(D1[:2000, 0].cpu(), D1[:2000, 1].cpu(), s=2, alpha=0.5)
axes[0].set_title('Real Data ($\\pi_1$)')
axes[0].set_aspect('equal')

axes[1].scatter(gen_A[:, 0], gen_A[:, 1], s=2, alpha=0.5, color='tab:blue')
axes[1].set_title('get_loss (built-in)')
axes[1].set_aspect('equal')

axes[2].scatter(gen_B[:, 0], gen_B[:, 1], s=2, alpha=0.5, color='tab:orange')
axes[2].set_title('Manual Loss')
axes[2].set_aspect('equal')

plt.suptitle('Generated Samples: Built-in vs Manual Loss', fontsize=14)
plt.tight_layout()
plt.show()

### Part 3 Observations

**Loss curves:** Both implementations produce nearly identical training loss curves, confirming that our manual loss

$$\mathcal{L} = \frac{1}{B}\sum_{i=1}^{B} \|(x_1^{(i)} - x_0^{(i)}) - v_\theta(x_t^{(i)}, t^{(i)})\|^2$$

is equivalent to the built-in `get_loss` method (which uses the same straight interpolation and uniform time sampling under the hood).

**Generated samples:** The two models produce visually indistinguishable samples — both capture the two-GMM target distribution accurately.

**Key insight:** The RF training objective is remarkably simple: sample random time $t$, interpolate, predict velocity, and minimize MSE. The `get_loss` method adds flexibility (different interpolation schemes, time weighting), but for the straight interpolation case, the manual implementation is straightforward.